In [1]:
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
import tqdm
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms,models
import torch.nn as nn
import torch.optim as optim

In [2]:
device=torch.device("cuda" if torch.cuda.is_available() else "CPU")
print(device)

cuda


In [3]:
# load  dataset
Dataset_path=Path(r"C:\Users\goura\OneDrive\Desktop\image_Similarity\visual-product-search\data\raw\Stanford_Online_Products")
train_df=pd.read_csv(Dataset_path/"Ebay_train.txt",sep=" ")
test_df=pd.read_csv(Dataset_path /"Ebay_test.txt",sep=" ")


In [4]:
img_transform=transforms.Compose([
    transforms.Resize((128,128)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [26]:
class dataset(Dataset):
    def __init__(self,dataframe,dir,transform=None):
        self.dataframe=dataframe
        self.dir=dir
        self.transform=transform
    def __len__(self):
        return len(self.dataframe)
    def __getitem__ (self,idx):
        row=self.dataframe.iloc[idx]
        image_path = self.dir / row["path"]
        images=Image.open(image_path).convert("RGB")
        labels=row["class_id"]
        if self.transform:
            images=self.transform(images)
        return images,labels,str(image_path)


In [27]:
train_d=dataset(dataframe=train_df,dir=Dataset_path,transform=img_transform)
test_d=dataset(dataframe=test_df,dir=Dataset_path,transform=img_transform)

In [28]:
train_loader=DataLoader(train_d,batch_size=16,shuffle=False,num_workers=0,pin_memory=True)
test_loader=DataLoader(test_d,batch_size=16,shuffle=False,num_workers=0,pin_memory=True)

In [29]:
# image, label = train_d[0]
# print(label)
# print(image.shape)

In [30]:
weights = models.ResNet50_Weights.DEFAULT
model = models.resnet50(weights=weights)

In [31]:
print(model)

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 

In [43]:
class FeatureExtractor(nn.Module):
    """
    Resnet50 Feature Extractor 
    Convert an Iput image to 2048 embedding vectors
    """
    def __init__(self):
        super().__init__()
        self.model=models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
        self.model.fc=nn.Identity()
    def forward(self,x):
        return self.model(x)
Model=FeatureExtractor().to(device)

In [33]:
# import time
# start = time.time()
# images, labels = next(iter(train_loader))
# print(f"Time: {time.time() - start:.2f} seconds")

In [34]:
# images=images.to(device)
# labels=labels.to(device)

In [35]:
# print(images.shape)
# print(labels.shape)

In [36]:
# with torch.no_grad():
#     embedding=model(images)
#     embeddings=embeddings.cpu().numpy()
#     print(embedding.shape)


In [37]:
# with torch.no_grad():
#     embeddings=model(images)
# print(embeddings.shape)

In [38]:
# embeddings=embeddings.cpu().numpy()

In [39]:
# embeddings.shape

In [40]:
# from pathlib import Path
# import numpy as np
# PROJECT_ROOT = Path(r"C:\Users\goura\OneDrive\Desktop\image_Similarity\visual-product-search")

# DATA_DIR = PROJECT_ROOT / "data"
# EMBEDDING_DIR = DATA_DIR / "embeddings"

# EMBEDDING_DIR.mkdir(parents=True, exist_ok=True)
# np.save(EMBEDDING_DIR / "embeddings.npy", embeddings)

In [41]:
from tqdm.auto import tqdm

In [45]:
# images, labels, paths = next(iter(train_loader))

# print(images.shape)
# print(labels.shape)
# print(type(paths))
# print(paths[0])

In [ ]:
all_emebddings=[]
all_labels=[]
all_paths=[]
Model.eval()
with torch.no_grad():
        for images, labels, paths in tqdm(train_loader):
                images,labels=images.to(device),labels.to(device)
                embeddings=Model(images)
                all_emebddings.append(embeddings.cpu())
                all_labels.extend(labels.cpu().numpy())
                all_paths.extend(paths)

  0%|          | 0/3722 [00:00<?, ?it/s]

In [62]:
print(len(all_labels))
print(len(all_paths))
all_emebddings

59551
59551


[tensor([[0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.1952, 0.0000],
         [0.3137, 0.0000, 0.0000,  ..., 0.0000, 0.1124, 0.0000],
         [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0268, 0.0000],
         ...,
         [0.2077, 0.0000, 0.2860,  ..., 0.0000, 0.0000, 0.0000],
         [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0543],
         [0.5225, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000]]),
 tensor([[0.2410, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0089],
         [0.7126, 0.0000, 0.0372,  ..., 0.0000, 0.0000, 0.0000],
         [0.0000, 0.0000, 0.3189,  ..., 0.0000, 0.0000, 0.0000],
         ...,
         [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.1328],
         [0.0348, 0.0000, 0.3342,  ..., 0.0000, 0.0000, 0.2498],
         [0.0134, 0.0000, 0.0172,  ..., 0.0000, 0.0000, 0.1195]]),
 tensor([[0.0000, 0.0000, 0.2661,  ..., 0.0000, 0.0000, 0.0716],
         [0.0000, 0.0000, 0.0000,  ..., 0.0046, 0.0000, 0.0790],
         [0.0000, 0.0000, 0.2934,  ..., 0.0000, 0.0000, 0.

In [64]:
all_emebddings=torch.cat(all_emebddings,dim=0).numpy()

In [67]:
print(all_emebddings.shape)

(59551, 2048)


In [68]:
all_labels=np.array(all_labels)

In [69]:
from pathlib import Path
import numpy as np
PROJECT_ROOT = Path(r"C:\Users\goura\OneDrive\Desktop\image_Similarity\visual-product-search")

DATA_DIR = PROJECT_ROOT / "data"
EMBEDDING_DIR = DATA_DIR / "embeddings"

EMBEDDING_DIR.mkdir(parents=True, exist_ok=True)
np.save(EMBEDDING_DIR / "train_embeddings.npy",all_emebddings)
np.save(EMBEDDING_DIR / "train_labels.npy",all_labels)

In [70]:
import pickle 
with open(EMBEDDING_DIR / "train_paths.pkl","wb") as f:
    pickle.dump(all_paths,f)


In [71]:
emb = np.load(EMBEDDING_DIR / "train_embeddings.npy")
labels = np.load(EMBEDDING_DIR / "train_labels.npy")

with open(EMBEDDING_DIR / "train_paths.pkl", "rb") as f:
    paths = pickle.load(f)

print(emb.shape)
print(labels.shape)
print(len(paths))

(59551, 2048)
(59551,)
59551
